<a href="https://colab.research.google.com/github/marciaspsilva/salesinsight-py/blob/feat%2Fpipeline-dados/salesinsight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Requisito Funcional 01 – Criação do dataset de vendas

In [1]:
import csv
import random
from datetime import datetime, timedelta

def gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos e grava em CSV."""
    random.seed(seed)

    produtos = [
        "Notebook",
        "Smartphone",
        "Tablet",
        "Monitor",
        "Teclado",
        "Mouse",
        "Headset"
    ]

    categorias = {
        "Notebook": "Computadores",
        "Smartphone": "Celulares",
        "Tablet": "Celulares",
        "Monitor": "Computadores",
        "Teclado": "Perifericos",
        "Mouse": "Perifericos",
        "Headset": "Perifericos"
    }

    precos = {
        "Notebook": 3500,
        "Smartphone": 2200,
        "Tablet": 1800,
        "Monitor": 1200,
        "Teclado": 250,
        "Mouse": 120,
        "Headset": 350
    }

    regioes = [
        "Sudeste",
        "Sul",
        "Nordeste",
        "Centro-Oeste",
        "Norte"
    ]

    data_inicio = datetime(2025, 1, 1)

    colunas = [
        "id_venda",
        "data_venda",
        "cliente",
        "produto",
        "categoria",
        "regiao",
        "quantidade",
        "preco_unitario"
    ]

    with open(caminho_csv, "w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=colunas)
        escritor.writeheader()

        for i in range(n_registros):
            produto = random.choice(produtos)
            categoria = categorias[produto]
            quantidade = random.randint(1, 10)
            preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)

            data = data_inicio + timedelta(days=random.randint(0, 364))
            data_txt = data.strftime("%Y-%m-%d")

            cliente = f"Cliente_{random.randint(1, 50):03d}"

            # --- sujeira proposital para a etapa de limpeza ---
            if random.random() < 0.05:
                quantidade = ""  # valor ausente

            if random.random() < 0.04:
                preco = ""  # valor ausente

            if random.random() < 0.06:
                produto = " " + produto + " "  # espacos extras

            if random.random() < 0.03:
                data_txt = "DATA INVALIDA"  # data invalida

            if random.random() < 0.10:
                cliente = random.choice([
                    cliente.upper().replace("_", "-"),
                    cliente + "!!",
                    " " + cliente,
                    cliente.replace("Cliente_", "cliente#"),
                ])

            escritor.writerow({
                "id_venda": i + 1,
                "data_venda": data_txt,
                "cliente": cliente,
                "produto": produto,
                "categoria": categoria,
                "regiao": random.choice(regioes),
                "quantidade": quantidade,
                "preco_unitario": preco,
            })

    print(f"Dataset gerado com {n_registros} registros em {caminho_csv}.")


gerar_dataset_vendas()

Dataset gerado com 200 registros em vendas.csv.


### Requisito Funcional 02 – Inspecionar e descrever os dados

In [2]:
import csv

def carregar_dataset_csv(arquivo):
    ### Carrega um arquivo CSV e retorna uma lista de dicionarios
    try:
        with open(arquivo, 'r', newline='', encoding='utf-8') as arquivo_csv:
            leitor = csv.DictReader(arquivo_csv)
            registros = list(leitor)
            return registros

    except FileNotFoundError:
        print(f"Arquivo '{arquivo}' não encontrado.")
    except Exception as e:
        print(f"Ocorreu um erro ao ler o arquivo CSV: {e}")
    finally:
        print("Processamento de leitura do csv finalizado.")

def inspecionar_dados(registros):
    ### Exibe as informacoes da lista de registros
    colunas = list(registros[0].keys()) if registros else []
    nulos = {coluna: 0 for coluna in colunas}
    for linha in registros:
        for coluna in colunas:
            if linha.get(coluna, "").strip() == "":
                nulos[coluna] += 1

    print("\n=== INSPECAO INICIAL DO DATASET ===")
    print(f"Total de registros: {len(registros)}")
    print(f"\nColunas: {colunas}")
    print(f"\nValores ausentes por coluna:\n{nulos}")
    print("\nPrimeiros registros:")
    for linha in registros[:5]:
        print(linha)


arquivo = 'vendas.csv'
registros = carregar_dataset_csv(arquivo)
inspecionar_dados(registros)

Processamento de leitura do csv finalizado.

=== INSPECAO INICIAL DO DATASET ===
Total de registros: 200

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Valores ausentes por coluna:
{'id_venda': 0, 'data_venda': 0, 'cliente': 0, 'produto': 0, 'categoria': 0, 'regiao': 0, 'quantidade': 10, 'preco_unitario': 4}

Primeiros registros:
{'id_venda': '1', 'data_venda': '2025-05-21', 'cliente': 'cliente#016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': '2', 'preco_unitario': '102.9'}
{'id_venda': '2', 'data_venda': '2025-09-16', 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}
{'id_venda': '3', 'data_venda': '2025-03-23', 'cliente': 'Cliente_045', 'produto': 'Tablet', 'categoria': 'Celulares', 'regiao': 'Centro-Oeste', 'quantidade': '1', 'preco_unitario': '1939.76'}
{'id_venda': '4', 'data_venda'

### Requisito Funcional 03 – Limpar e tratar os dados (datetime e regex)

In [3]:
import re
from datetime import datetime

def limpar_dados(registros):
    """
    Limpa e trata a lista de registros de vendas.
    Retorna: (registros_limpos, relatorio), onde relatorio é um dicionario
    com as contagens de registros iniciais, removidos e finais.
    """
    relatorio = {"iniciais": len(registros), "removidos_data": 0,
                 "removidos_nulos": 0, "finais": 0}
    padrao_cliente = re.compile(r"^Cliente_\d{3}$", flags=re.IGNORECASE)

    limpos = []
    registros_removidos = []
    for linha in registros:

        registros_removidos.append(linha)

        # 1. remover espacos extras nas colunas de texto
        for chave in ("cliente", "produto", "categoria", "regiao"):
            linha[chave] = linha[chave].strip()

        # 2. converter data_venda e descartar datas invalidas
        try:
            linha["data_venda"] = datetime.strptime(linha["data_venda"], "%Y-%m-%d")
        except ValueError:
            relatorio["removidos_data"] += 1
            registros_removidos.append(linha)
            continue

        # 3. descartar nulos em quantidade e preco_unitario
        if linha["quantidade"] == "" or linha["preco_unitario"] == "":
            relatorio["removidos_nulos"] += 1
            registros_removidos.append(linha)
            continue

        # 4. ajustar os tipos numericos
        linha["quantidade"] = int(float(linha["quantidade"]))
        linha["preco_unitario"] = float(linha["preco_unitario"])

        # 5. padronizar o nome do cliente com re.sub()
        nome_limpo = re.sub(r"[^A-Za-z0-9_]", "", linha["cliente"].strip())
        linha["cliente"] = nome_limpo
        linha["cliente_fora_do_padrao"] = padrao_cliente.match(nome_limpo) is None

        limpos.append(linha)

    # 6. montar e imprimir o relatorio de limpeza
    relatorio["finais"] = len(limpos)
    print("\n=== RELATORIO DE LIMPEZA ===")
    print(relatorio)
    print("\n=== REGISTROS REMOVIDOS ===")
    for registro in registros_removidos:
        print(registro)
    return limpos, relatorio

registros_limpos, relatorio_limpeza = limpar_dados(registros)


=== RELATORIO DE LIMPEZA ===
{'iniciais': 200, 'removidos_data': 4, 'removidos_nulos': 13, 'finais': 183}

=== REGISTROS REMOVIDOS ===
{'id_venda': '1', 'data_venda': datetime.datetime(2025, 5, 21, 0, 0), 'cliente': 'cliente016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': 2, 'preco_unitario': 102.9, 'cliente_fora_do_padrao': True}
{'id_venda': '2', 'data_venda': datetime.datetime(2025, 9, 16, 0, 0), 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}
{'id_venda': '2', 'data_venda': datetime.datetime(2025, 9, 16, 0, 0), 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}
{'id_venda': '3', 'data_venda': datetime.datetime(2025, 3, 23, 0, 0), 'cliente': 'Cliente_045', 'produto': 'Tablet', 'categoria': 'Celulares', 'regiao': 'Centro-Oeste', 'quantidade': 1, 'preco_unita